# Road PCI Intelligence: Pavement Distress Data Exploration
### Automated Pavement Condition Index (PCI) Pipeline

This notebook performs an exploratory data analysis (EDA) and annotation schema audit across four candidate road distress datasets mounted from Kaggle:
1. **`farzadnekouei/pothole-image-segmentation-dataset`**: Pothole segmentation with YOLOv8 polygon masks.
2. **`rukiyeaydn/road-crack-dataset`**: Road crack segmentation with paired PNG binary masks.
3. **`sj26717/crack500`**: Pavement crack segmentation masks from the CRACK500 benchmark.
4. **`juusos/rdd2022es`**: Extended Severity RDD2022 covering the 4 distress classes (longitudinal, transverse, alligator cracks, potholes).

**Objectives:**
- Discover folder layouts (2 levels deep) and file extension counts dynamically.
- Compute image resolution distributions (min, max, mean, most common).
- Inspect and visualize image/mask pairs and verify unique pixel values.
- Parse RDD2022ES multi-class annotations and render bounding boxes on sample roadway images.
- Complete the structured **Observations** questionnaire for downstream model design.


In [ ]:
import os
import glob
import re
import yaml
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import cv2

# Plotting configuration
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10
%matplotlib inline

INPUT_ROOT = Path('/kaggle/input')

print(f"Checking {INPUT_ROOT}:")
if INPUT_ROOT.exists():
    available_datasets = sorted([p.name for p in INPUT_ROOT.iterdir() if p.is_dir()])
    print(f"Found {len(available_datasets)} datasets mounted:")
    for ds in available_datasets:
        print(f"  • {ds}")
else:
    print("Warning: /kaggle/input not found. Running in local test mode.")
    available_datasets = []


In [ ]:
def print_folder_tree(start_path, max_depth=2, max_items_per_dir=6):
    """Prints a directory tree up to max_depth levels deep."""
    start_path = Path(start_path)
    if not start_path.exists():
        print(f"Path does not exist: {start_path}")
        return

    def _walk(current_path, depth):
        if depth > max_depth:
            return
        indent = "    " * depth
        try:
            items = sorted(list(current_path.iterdir()), key=lambda x: (not x.is_dir(), x.name.lower()))
        except PermissionError:
            print(f"{indent}[Access Denied]")
            return

        dirs = [item for item in items if item.is_dir()]
        files = [item for item in items if not item.is_dir()]

        for d in dirs[:max_items_per_dir]:
            print(f"{indent}📁 {d.name}/")
            _walk(d, depth + 1)
        if len(dirs) > max_items_per_dir:
            print(f"{indent}   ... ({len(dirs) - max_items_per_dir} more directories)")

        for f in files[:max_items_per_dir]:
            size_kb = f.stat().st_size / 1024
            print(f"{indent}📄 {f.name} ({size_kb:.1f} KB)")
        if len(files) > max_items_per_dir:
            print(f"{indent}   ... ({len(files) - max_items_per_dir} more files)")

    print(f"📂 {start_path.name}/")
    _walk(start_path, 1)


def audit_files_and_extensions(dataset_path):
    """Counts total files and summarizes file extensions."""
    dataset_path = Path(dataset_path)
    ext_counter = Counter()
    total_files = 0
    total_size_bytes = 0

    for root, _, files in os.walk(dataset_path):
        for f in files:
            total_files += 1
            ext = Path(f).suffix.lower()
            ext_counter[ext if ext else '[no_ext]'] += 1
            try:
                total_size_bytes += os.path.getsize(os.path.join(root, f))
            except OSError:
                pass

    return {
        'total_files': total_files,
        'total_size_mb': total_size_bytes / (1024 * 1024),
        'extensions': dict(ext_counter.most_common())
    }


def compute_image_size_stats(image_paths, sample_limit=600):
    """Computes min, max, mean, and mode image dimensions using PIL without loading full pixel buffers."""
    if not image_paths:
        return {}
    
    sample = image_paths[:sample_limit]
    dimensions = []
    
    for p in sample:
        try:
            with Image.open(p) as img:
                dimensions.append(img.size) # (width, height)
        except Exception:
            continue
            
    if not dimensions:
        return {}
        
    widths = [w for w, h in dimensions]
    heights = [h for w, h in dimensions]
    res_counts = Counter(dimensions)
    
    return {
        'count_sampled': len(dimensions),
        'min_width': min(widths),
        'max_width': max(widths),
        'mean_width': round(np.mean(widths), 1),
        'min_height': min(heights),
        'max_height': max(heights),
        'mean_height': round(np.mean(heights), 1),
        'most_common_resolutions': res_counts.most_common(3)
    }


In [ ]:
print("=" * 80)
print("AUDIT: DATASET FOLDER TREES AND FILE EXTENSIONS")
print("=" * 80)

dataset_dirs = [p for p in INPUT_ROOT.iterdir() if p.is_dir()] if INPUT_ROOT.exists() else []

audit_summaries = {}
all_image_paths = defaultdict(list)

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

for ds_dir in dataset_dirs:
    print(f"\n--- Dataset: {ds_dir.name} ---")
    print_folder_tree(ds_dir, max_depth=2, max_items_per_dir=6)
    
    audit = audit_files_and_extensions(ds_dir)
    audit_summaries[ds_dir.name] = audit
    print(f"Total files: {audit['total_files']} | Total size: {audit['total_size_mb']:.2f} MB")
    print(f"File extensions: {audit['extensions']}")
    
    # Collect all image paths
    for root, _, files in os.walk(ds_dir):
        for f in files:
            if Path(f).suffix.lower() in IMAGE_EXTS:
                all_image_paths[ds_dir.name].append(Path(root) / f)
                
    print(f"Total images detected: {len(all_image_paths[ds_dir.name])}")


In [ ]:
print("=" * 80)
print("IMAGE RESOLUTION AND SIZE DISTRIBUTION")
print("=" * 80)

size_stats_records = []

for ds_name, img_paths in all_image_paths.items():
    stats = compute_image_size_stats(img_paths, sample_limit=500)
    if stats:
        common_str = ", ".join([f"{w}x{h} ({cnt})" for (w, h), cnt in stats['most_common_resolutions']])
        size_stats_records.append({
            'Dataset': ds_name,
            'Total Images': len(img_paths),
            'Sampled': stats['count_sampled'],
            'Min (WxH)': f"{stats['min_width']}x{stats['min_height']}",
            'Max (WxH)': f"{stats['max_width']}x{stats['max_height']}",
            'Mean (WxH)': f"{stats['mean_width']}x{stats['mean_height']}",
            'Top Common Resolutions': common_str
        })

df_size_stats = pd.DataFrame(size_stats_records)
display(df_size_stats)


In [ ]:
def display_dataset_grid(image_paths, dataset_name, n=6):
    """Displays n sample images in a grid with filename and resolution info."""
    if not image_paths:
        print(f"No images found for {dataset_name}")
        return
    
    # Filter out obvious mask images if possible for general preview
    clean_images = [p for p in image_paths if 'mask' not in p.stem.lower()]
    sample = clean_images[:n] if len(clean_images) >= n else image_paths[:n]
    
    cols = 3
    rows = int(np.ceil(len(sample) / cols))
    
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
    fig.suptitle(f"Sample Roadway Images: {dataset_name}", fontsize=14, weight='bold', y=0.98)
    axes = np.array(axes).reshape(-1)
    
    for idx, img_path in enumerate(sample):
        try:
            with Image.open(img_path) as img:
                img_rgb = img.convert('RGB')
                w, h = img.size
                axes[idx].imshow(img_rgb)
                axes[idx].set_title(f"{img_path.name[:25]}...\n({w}x{h})", fontsize=9)
                axes[idx].axis('off')
        except Exception as e:
            axes[idx].text(0.5, 0.5, f"Error: {e}", ha='center')
            axes[idx].axis('off')
            
    for j in range(len(sample), len(axes)):
        axes[j].axis('off')
        
    plt.tight_layout()
    plt.show()

for ds_name, paths in all_image_paths.items():
    display_dataset_grid(paths, ds_name, n=6)


## Deep Dive 1: Semantic Mask Datasets (`road-crack-dataset` & `crack500`)
We now inspect datasets providing pixel-level segmentation masks:
- Discover image-to-mask pairings.
- Load 6 image/mask pairs and visualize them side-by-side.
- Inspect the unique pixel values present in the masks to confirm binary ($0, 255$ or $0, 1$) vs multi-class encoding.


In [ ]:
def explore_road_crack_dataset(ds_path):
    print("=" * 80)
    print("INSPECTING: road-crack-dataset (rukiyeaydn/road-crack-dataset)")
    print("=" * 80)
    
    ds_path = Path(ds_path)
    all_files = list(ds_path.rglob('*'))
    
    # Find image and mask pairs:
    # Pattern: image.jpg and image_mask.png
    mask_files = [f for f in all_files if f.name.endswith('_mask.png') or '_mask' in f.stem]
    print(f"Total masks found: {len(mask_files)}")
    
    pairs = []
    for mask_p in mask_files:
        # Expected image file name without _mask
        base_name = mask_p.name.replace('_mask.png', '')
        possible_imgs = list(mask_p.parent.glob(f"{base_name}*"))
        img_candidates = [p for p in possible_imgs if p != mask_p and p.suffix.lower() in {'.jpg', '.png', '.jpeg'}]
        if img_candidates:
            pairs.append((img_candidates[0], mask_p))
            
    print(f"Successfully paired {len(pairs)} image/mask sets.")
    
    if not pairs:
        print("Fallback: Attempting directory-level matching...")
        img_dir = ds_path / 'images'
        mask_dir = ds_path / 'masks'
        if img_dir.exists() and mask_dir.exists():
            for img_p in list(img_dir.glob('*.jpg'))[:10]:
                mask_p = mask_dir / f"{img_p.stem}.png"
                if mask_p.exists():
                    pairs.append((img_p, mask_p))
                    
    # Inspect unique pixel values across first 10 masks
    unique_values_all = set()
    for _, mask_p in pairs[:10]:
        mask_arr = np.array(Image.open(mask_p))
        unique_values_all.update(np.unique(mask_arr))
        
    print(f"\nUnique pixel values found in masks: {sorted(list(unique_values_all))}")
    if unique_values_all == {0, 255} or unique_values_all == {0, 1}:
        print("-> Confirmed BINARY segmentation mask format.")
    else:
        print(f"-> Mask contains values: {sorted(list(unique_values_all))}")
        
    # Visualize 6 pairs side-by-side
    sample_pairs = pairs[:6]
    if sample_pairs:
        fig, axes = plt.subplots(len(sample_pairs), 2, figsize=(10, 3.5 * len(sample_pairs)))
        fig.suptitle("road-crack-dataset: Image vs Ground Truth Mask", fontsize=14, weight='bold')
        
        for i, (img_p, mask_p) in enumerate(sample_pairs):
            img = Image.open(img_p).convert('RGB')
            mask = Image.open(mask_p)
            mask_arr = np.array(mask)
            
            axes[i, 0].imshow(img)
            axes[i, 0].set_title(f"Image: {img_p.name[:25]} ({img.size[0]}x{img.size[1]})", fontsize=9)
            axes[i, 0].axis('off')
            
            axes[i, 1].imshow(mask_arr, cmap='gray')
            axes[i, 1].set_title(f"Mask: {mask_p.name[:25]} | Unique: {np.unique(mask_arr)}", fontsize=9)
            axes[i, 1].axis('off')
            
        plt.tight_layout()
        plt.show()

# Run for road-crack-dataset if mounted
for d in dataset_dirs:
    if 'road-crack' in d.name.lower():
        explore_road_crack_dataset(d)


In [ ]:
def explore_crack500_dataset(ds_path):
    print("=" * 80)
    print("INSPECTING: crack500 (sj26717/crack500)")
    print("=" * 80)
    
    ds_path = Path(ds_path)
    all_files = list(ds_path.rglob('*'))
    
    # Look for masks in Annotations/masks or similar
    mask_files = [f for f in all_files if 'mask' in f.name.lower() and f.suffix.lower() == '.png']
    print(f"Total masks found: {len(mask_files)}")
    
    pairs = []
    for mask_p in mask_files:
        # Mask name: 20160222_080850_mask.png -> Image: 20160222_080850.jpg or .png
        stem = mask_p.stem.replace('_mask', '')
        # Search for corresponding image
        matches = list(ds_path.rglob(f"{stem}.*"))
        img_candidates = [m for m in matches if m != mask_p and m.suffix.lower() in {'.jpg', '.png', '.jpeg'}]
        if img_candidates:
            pairs.append((img_candidates[0], mask_p))
            
    print(f"Paired {len(pairs)} image/mask sets.")
    
    if pairs:
        # Inspect pixel values
        unique_vals = set()
        for _, mask_p in pairs[:10]:
            mask_arr = np.array(Image.open(mask_p))
            unique_vals.update(np.unique(mask_arr))
            
        print(f"Unique pixel values in CRACK500 masks: {sorted(list(unique_vals))}")
        
        # Visualize 6 pairs
        sample_pairs = pairs[:6]
        fig, axes = plt.subplots(len(sample_pairs), 2, figsize=(10, 3.5 * len(sample_pairs)))
        fig.suptitle("CRACK500: Image vs Ground Truth Mask", fontsize=14, weight='bold')
        
        for i, (img_p, mask_p) in enumerate(sample_pairs):
            img = Image.open(img_p).convert('RGB')
            mask = Image.open(mask_p)
            mask_arr = np.array(mask)
            
            axes[i, 0].imshow(img)
            axes[i, 0].set_title(f"Image: {img_p.name} ({img.size[0]}x{img.size[1]})", fontsize=9)
            axes[i, 0].axis('off')
            
            axes[i, 1].imshow(mask_arr, cmap='gray')
            axes[i, 1].set_title(f"Mask: {mask_p.name} | Unique: {np.unique(mask_arr)}", fontsize=9)
            axes[i, 1].axis('off')
            
        plt.tight_layout()
        plt.show()

for d in dataset_dirs:
    if 'crack500' in d.name.lower():
        explore_crack500_dataset(d)


## Deep Dive 2: YOLOv8 Pothole Segmentation (`pothole-image-segmentation-dataset`)
The pothole dataset uses YOLOv8 segmentation annotations:
- Normalised polygon coordinates (`class_id x1 y1 x2 y2 ... xn yn`).
- Here we:
  1. Inspect `data.yaml` to confirm class names.
  2. Parse the polygon coordinates for 6 sample images.
  3. Rasterize polygon contours into binary pixel masks using `cv2.fillPoly`.
  4. Display the original roadway image side-by-side with the generated pixel mask, verifying unique pixel values ($0, 255$).


In [ ]:
def explore_pothole_segmentation(ds_path):
    print("=" * 80)
    print("INSPECTING: Pothole Image Segmentation Dataset (farzadnekouei)")
    print("=" * 80)
    
    ds_path = Path(ds_path)
    
    # Check data.yaml
    yaml_files = list(ds_path.rglob('*.yaml'))
    if yaml_files:
        print(f"Found config: {yaml_files[0].name}")
        try:
            with open(yaml_files[0], 'r') as f:
                cfg = yaml.safe_load(f)
            print(f"Classes: {cfg.get('names', 'N/A')} | Number of classes: {cfg.get('nc', 'N/A')}")
        except Exception as e:
            print(f"Could not parse yaml: {e}")
            
    # Pair images and label .txt files
    img_files = [f for f in ds_path.rglob('*.jpg') if not f.name.startswith('.')]
    pairs = []
    
    for img_p in img_files:
        # YOLO structure: images/xxx.jpg -> labels/xxx.txt
        label_p = Path(str(img_p).replace('/images/', '/labels/').replace('\\images\\', '\\labels\\')).with_suffix('.txt')
        if label_p.exists():
            pairs.append((img_p, label_p))
            
    print(f"Paired {len(pairs)} images with YOLO segmentation label files.")
    
    def parse_yolo_polygon_mask(img_shape, label_file):
        """Rasterizes YOLO normalized polygon points to a binary mask."""
        h, w = img_shape[:2]
        mask = np.zeros((h, w), dtype=np.uint8)
        
        if not label_file.exists():
            return mask
            
        with open(label_file, 'r') as f:
            lines = f.readlines()
            
        for line in lines:
            parts = line.strip().split()
            if len(parts) > 4: # Polygon points
                class_id = int(parts[0])
                coords = [float(x) for x in parts[1:]]
                # Group into (x, y) pairs and denormalize
                pts = []
                for i in range(0, len(coords), 2):
                    px = int(coords[i] * w)
                    py = int(coords[i + 1] * h)
                    pts.append([px, py])
                pts = np.array(pts, dtype=np.int32).reshape((-1, 1, 2))
                cv2.fillPoly(mask, [pts], color=255)
        return mask

    sample_pairs = pairs[:6]
    if sample_pairs:
        fig, axes = plt.subplots(len(sample_pairs), 2, figsize=(10, 3.5 * len(sample_pairs)))
        fig.suptitle("Pothole Dataset: Image vs Rasterized Polygon Mask", fontsize=14, weight='bold')
        
        for i, (img_p, label_p) in enumerate(sample_pairs):
            img_bgr = cv2.imread(str(img_p))
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            mask = parse_yolo_polygon_mask(img_rgb.shape, label_p)
            
            axes[i, 0].imshow(img_rgb)
            axes[i, 0].set_title(f"Image: {img_p.name} ({img_rgb.shape[1]}x{img_rgb.shape[0]})", fontsize=9)
            axes[i, 0].axis('off')
            
            axes[i, 1].imshow(mask, cmap='gray')
            axes[i, 1].set_title(f"Rasterized Mask | Unique values: {np.unique(mask)}", fontsize=9)
            axes[i, 1].axis('off')
            
        plt.tight_layout()
        plt.show()

for d in dataset_dirs:
    if 'pothole' in d.name.lower():
        explore_pothole_segmentation(d)


## Deep Dive 3: Multi-Class Road Distress (`juusos/rdd2022es`)
The `rdd2022es` dataset contains the **4 core pavement distress classes** (Longitudinal, Transverse, Alligator cracks, and Potholes) extended with severity levels:
- `0`: Longitudinal Crack (minor)
- `1`: Longitudinal Crack (wide / severe)
- `2`: Transverse Crack (minor)
- `3`: Transverse Crack (wide / severe)
- `4`: Alligator Crack (minor)
- `5`: Alligator Crack (sunken / severe)
- `6`: Pothole (minor)
- `7`: Pothole (deep / severe)

Below, we parse annotations, calculate box counts across distress classes, and render 6 roadway images with labeled bounding boxes.


In [ ]:
RDD_CLASS_MAP = {
    0: 'Longitudinal Crack',
    1: 'Longitudinal Crack (Wide)',
    2: 'Transverse Crack',
    3: 'Transverse Crack (Wide)',
    4: 'Alligator Crack',
    5: 'Alligator Crack (Sunken)',
    6: 'Pothole',
    7: 'Pothole (Deep)'
}

CLASS_COLORS = {
    0: '#1f77b4', 1: '#aec7e8',
    2: '#ff7f0e', 3: '#ffbb78',
    4: '#2ca02c', 5: '#98df8a',
    6: '#d62728', 7: '#ff9896'
}

def explore_rdd2022es_dataset(ds_path, sample_parse_limit=500):
    print("=" * 80)
    print("INSPECTING: RDD2022ES Multi-Class Distress (juusos/rdd2022es)")
    print("=" * 80)
    
    ds_path = Path(ds_path)
    txt_annotations = list(ds_path.rglob('*.txt'))
    # Filter out readmes/configs
    txt_annotations = [f for f in txt_annotations if not f.name.startswith(('README', 'classes', 'data'))]
    print(f"Total annotation files detected: {len(txt_annotations)}")
    
    class_box_counts = Counter()
    image_annotation_pairs = []
    
    # Parse annotations
    for txt_p in txt_annotations[:sample_parse_limit]:
        img_candidate = txt_p.with_suffix('.jpg')
        if not img_candidate.exists():
            img_candidate = txt_p.with_suffix('.png')
            
        boxes = []
        with open(txt_p, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(float(parts[0]))
                    xc, yc, w, h = [float(x) for x in parts[1:5]]
                    boxes.append((cls_id, xc, yc, w, h))
                    class_box_counts[cls_id] += 1
                    
        if img_candidate.exists() and len(boxes) > 0:
            image_annotation_pairs.append((img_candidate, boxes))
            
    print(f"Parsed {len(txt_annotations[:sample_parse_limit])} files. Bounding box distribution:")
    for cls_id in sorted(RDD_CLASS_MAP.keys()):
        cnt = class_box_counts[cls_id]
        print(f"  [{cls_id}] {RDD_CLASS_MAP[cls_id]:<28}: {cnt:>5} boxes")
        
    # Plot class distribution bar chart
    fig, ax = plt.subplots(figsize=(10, 4))
    classes_listed = [RDD_CLASS_MAP[i] for i in sorted(RDD_CLASS_MAP.keys())]
    counts_listed = [class_box_counts[i] for i in sorted(RDD_CLASS_MAP.keys())]
    ax.barh(classes_listed, counts_listed, color='#1f77b4')
    ax.set_xlabel('Bounding Box Count (Sampled)')
    ax.set_title('RDD2022ES Road Distress & Severity Distribution', weight='bold')
    plt.tight_layout()
    plt.show()
    
    # Visualize 6 images with bounding boxes drawn
    sample_to_plot = image_annotation_pairs[:6]
    if sample_to_plot:
        cols = 3
        rows = 2
        fig, axes = plt.subplots(rows, cols, figsize=(16, 10))
        axes = axes.reshape(-1)
        fig.suptitle("RDD2022ES: Sample Road Distress Annotations", fontsize=15, weight='bold')
        
        for idx, (img_p, boxes) in enumerate(sample_to_plot):
            img = Image.open(img_p).convert('RGB')
            w_img, h_img = img.size
            axes[idx].imshow(img)
            
            for (cls_id, xc, yc, bw, bh) in boxes:
                # Convert normalized xywh to pixel coords
                xmin = (xc - bw / 2) * w_img
                ymin = (yc - bh / 2) * h_img
                box_w = bw * w_img
                box_h = bh * h_img
                
                color = CLASS_COLORS.get(cls_id, 'red')
                rect = patches.Rectangle((xmin, ymin), box_w, box_h, linewidth=2, edgecolor=color, facecolor='none')
                axes[idx].add_patch(rect)
                
                label_text = RDD_CLASS_MAP.get(cls_id, str(cls_id))
                axes[idx].text(xmin, max(ymin - 4, 10), label_text, color='white', fontsize=7,
                               bbox=dict(boxstyle='square,pad=0.2', facecolor=color, alpha=0.8, edgecolor='none'))
                
            axes[idx].set_title(f"{img_p.name} ({len(boxes)} defects)", fontsize=9)
            axes[idx].axis('off')
            
        plt.tight_layout()
        plt.show()

for d in dataset_dirs:
    if 'rdd2022' in d.name.lower():
        explore_rdd2022es_dataset(d)


In [ ]:
synthesis_data = [
    {
        'Dataset': 'farzadnekouei/pothole-image-segmentation-dataset',
        'Annotation Format': 'YOLOv8 Polygon (.txt)',
        'Defect Types': 'Potholes only',
        'Pixel-Level Masks?': 'Convertible via cv2.fillPoly',
        'Size on Disk': '~60 MB',
        'Role in Pipeline': 'High-precision pothole segmenter & surface area quantification'
    },
    {
        'Dataset': 'rukiyeaydn/road-crack-dataset',
        'Annotation Format': 'Binary PNG Masks (_mask.png)',
        'Defect Types': 'Road Cracks (General)',
        'Pixel-Level Masks?': 'Direct (0 / 255 values)',
        'Size on Disk': '~47 MB',
        'Role in Pipeline': 'Clean binary crack baseline for U-Net / FPN training'
    },
    {
        'Dataset': 'sj26717/crack500',
        'Annotation Format': 'Binary PNG Masks (.png)',
        'Defect Types': 'Pavement Cracks (Complex texture)',
        'Pixel-Level Masks?': 'Direct (0 / 255 values)',
        'Size on Disk': '~498 MB',
        'Role in Pipeline': 'Benchmark generalization test set for fine crack structures'
    },
    {
        'Dataset': 'juusos/rdd2022es',
        'Annotation Format': 'YOLO Bounding Boxes (.txt)',
        'Defect Types': 'Longitudinal, Transverse, Alligator Cracks, Potholes (+Severity)',
        'Pixel-Level Masks?': 'Bounding boxes (requires Box-to-Mask or multi-stage pipeline)',
        'Size on Disk': '~2.4 GB',
        'Role in Pipeline': 'Multi-class distress classifier and severity assigner'
    }
]

df_synthesis = pd.DataFrame(synthesis_data)
print("=" * 80)
print("DATASET COMPARISON & PIPELINE INTEGRATION MATRIX")
print("=" * 80)
display(df_synthesis)


## Observations & Key Questions

Please review the outputs from this notebook and record your answers to the following project design questions:

1. **Image Resolutions & Aspect Ratios**:
   - What are the predominant image resolutions across each dataset?
   - Do images need resizing / letterboxing (e.g. to $512 \times 512$ or $640 \times 640$) before training?
2. **Label Formats**:
   - Which datasets provide ready-to-train raster masks (`0` and `255`), and which provide polygon coordinates?
   - For YOLO polygons, does `cv2.fillPoly` cleanly reconstruct the defect boundaries?
3. **Four Target Distress Classes**:
   - Which datasets cover each of the 4 distress categories:
     - Longitudinal Cracks:
     - Transverse Cracks:
     - Alligator Cracks:
     - Potholes:
4. **Class Imbalance & Annotation Quality**:
   - In `rdd2022es`, which distress classes are most frequent versus underrepresented?
   - Are there background artifacts (e.g., shadows, lane markings, oil stains) that could trigger false positives?
5. **Next Architecture Decision**:
   - Should we build a **two-stage pipeline** (Stage 1: Multi-class distress detection / classification via `rdd2022es`, Stage 2: Precision segmentation via `road-crack-dataset` and `pothole-segmentation`) or rasterize bounding boxes into pseudo-masks?
